# 213 — Chunk Documents

Chunks each section's verbatim text (extracted by notebook 211 into `sections.csv`) into
200-400 token segments for use in vector search. Each chunk belongs to exactly one section.

Pre-body pages (cover, ToC) that precede the first section are chunked from the raw PDF
and assigned to a synthetic preamble section.

| Output | Description |
|---|---|
| `chunks.csv` | All chunks across all documents |

Token estimate: 1 token ≈ 4 characters. Target chunk size: ~1 200 characters (≈300 tokens).

Re-runnable: yes.


In [1]:
import sys, os, yaml
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.document.pdf_utils import extract_pdf_pages
from src.document.config    import load_document_config

LAYER2_DIR       = project_root / 'data' / 'layer_2'
INTERMEDIATE_DIR = LAYER2_DIR / 'intermediate'
PDF_DIR          = LAYER2_DIR / 'regulatory_documents'
CONFIG_PATH      = LAYER2_DIR / 'document_config.yaml'

TARGET_CHARS = 1200   # ~300 tokens
MIN_CHARS    =  600   # ~150 tokens
MAX_CHARS    = 2000   # ~500 tokens

config = load_document_config(CONFIG_PATH)
docs   = config['documents']

print(f'Layer 2 dir : {LAYER2_DIR}')
print(f'Documents   : {len(docs)}')


Layer 2 dir : /Users/emillaurencepastor/Documents/github/graphrag-finserv-compliance/data/layer_2
Documents   : 1


## Step 1 — Load sections (includes verbatim text from notebook 211)


In [2]:
sections_df = pd.read_csv(LAYER2_DIR / 'sections.csv')
print(f'Sections loaded: {len(sections_df)}')

if 'text' not in sections_df.columns or sections_df['text'].isna().all():
    raise RuntimeError(
        "sections.csv has no 'text' column. Re-run notebook 211 to extract verbatim section text."
    )

filled = sections_df['text'].notna().sum()
print(f"Sections with verbatim text: {filled} / {len(sections_df)}")
print(sections_df[['regulation_id', 'section_id', 'section_type']].head(10).to_string())


Sections loaded: 30
Sections with verbatim text: 30 / 30
  regulation_id   section_id section_type
0       APS-220   APS-220-S1         core
1       APS-220   APS-220-S2         core
2       APS-220   APS-220-S3         core
3       APS-220   APS-220-S4         core
4       APS-220   APS-220-S5         core
5       APS-220   APS-220-S6         core
6       APS-220   APS-220-S7         core
7       APS-220   APS-220-S8         core
8       APS-220   APS-220-S9         core
9       APS-220  APS-220-S10         core


## Step 2 — Chunking functions


In [3]:
def chunk_text(text: str) -> list:
    """Split a plain text string into chunks of ~TARGET_CHARS.

    Splits on double-newline paragraph boundaries. Returns a list of text strings.
    """
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    chunks       = []
    current_text = ''

    def _flush():
        nonlocal current_text
        if current_text.strip():
            chunks.append(current_text.strip())
        current_text = ''

    for para in paragraphs:
        if len(para) > MAX_CHARS:
            for line in [ln.strip() for ln in para.splitlines() if ln.strip()]:
                if len(current_text) + len(line) + 1 <= MAX_CHARS:
                    current_text = (current_text + '\n' + line).strip()
                else:
                    if len(current_text) >= MIN_CHARS:
                        _flush()
                    current_text = line
            continue

        joined = (current_text + '\n\n' + para).strip() if current_text else para
        if len(joined) > MAX_CHARS and len(current_text) >= MIN_CHARS:
            _flush()
            current_text = para
        else:
            current_text = joined

    _flush()
    return chunks


def chunk_pages(pages: list) -> list:
    """Chunk a list of (page_num, text) tuples — used only for preamble pages.

    Flushes the current chunk whenever the page number changes, guaranteeing
    that every PDF page produces at least one chunk with correct page attribution.
    No chunk will silently span a page boundary.

    Returns a list of dicts: {text, source_page_start, source_page_end}.
    """
    tagged = []
    for page_num, page_text in pages:
        for para in page_text.split('\n\n'):
            para = para.strip()
            if para:
                tagged.append((page_num, para))

    chunks             = []
    current_text       = ''
    current_page_start = None
    current_page_end   = None
    prev_page_num      = None

    def _flush():
        nonlocal current_text, current_page_start, current_page_end
        if current_text.strip():
            chunks.append({
                'text':              current_text.strip(),
                'source_page_start': current_page_start,
                'source_page_end':   current_page_end,
            })
        current_text       = ''
        current_page_start = None
        current_page_end   = None

    for page_num, para in tagged:
        # Flush on page change — guarantees every page has its own chunk(s)
        if prev_page_num is not None and page_num != prev_page_num:
            _flush()

        prev_page_num = page_num

        if current_page_start is None:
            current_page_start = page_num

        if len(para) > MAX_CHARS:
            for line in [ln.strip() for ln in para.splitlines() if ln.strip()]:
                if len(current_text) + len(line) + 1 <= MAX_CHARS:
                    current_text       = (current_text + '\n' + line).strip()
                    current_page_end   = page_num
                else:
                    if len(current_text) >= MIN_CHARS:
                        _flush()
                    current_text       = line
                    current_page_start = page_num
                    current_page_end   = page_num
            continue

        joined = (current_text + '\n\n' + para).strip() if current_text else para
        if len(joined) > MAX_CHARS and len(current_text) >= MIN_CHARS:
            _flush()
            current_text       = para
            current_page_start = page_num
            current_page_end   = page_num
        else:
            current_text     = joined
            current_page_end = page_num

    _flush()
    return chunks


## Step 3 — Process all documents

In [4]:
all_chunks = []

for doc in docs:
    rid      = doc['regulation_id']
    pdf_path = PDF_DIR / doc['pdf_path']

    doc_sections = (
        sections_df[sections_df['regulation_id'] == rid].copy()
        if 'regulation_id' in sections_df.columns
        else sections_df.copy()
    )

    print(f'\n{rid}: {len(doc_sections)} sections')

    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF not found: {pdf_path}')

    pages        = extract_pdf_pages(pdf_path)   # [(page_num, text), ...]
    pages_by_num = {pn: text for pn, text in pages}
    total_pages  = max(pages_by_num) if pages_by_num else 0
    print(f'  Pages: {total_pages} | Characters: {sum(len(t) for t in pages_by_num.values()):,}')

    # Check for gaps in pdfplumber extraction
    all_page_nums = set(pages_by_num)
    missing_pdf   = set(range(1, total_pages + 1)) - all_page_nums
    if missing_pdf:
        print(f'  ⚠  pdfplumber returned no text for pages: {sorted(missing_pdf)}')

    # Determine preamble range
    page_starts  = doc_sections['source_page_start'].dropna().astype(int)
    body_start   = int(page_starts.min()) if not page_starts.empty else 1
    preamble_sid = f'{rid}-PREAMBLE'

    chunk_records    = []
    global_chunk_idx = 0

    # --- Preamble: one chunk per page, stub if no text ---
    if body_start > 1:
        for pn in range(1, body_start):
            page_text  = pages_by_num.get(pn, '').strip()
            raw_chunks = chunk_pages([(pn, page_text)]) if page_text else []

            if raw_chunks:
                for chunk in raw_chunks:
                    chunk_records.append({
                        'chunk_id':          f'{rid}-CHUNK-{global_chunk_idx + 1:04d}',
                        'section_id':        preamble_sid,
                        'text':              chunk['text'],
                        'token_count':       len(chunk['text']) // 4,
                        'chunk_index':       global_chunk_idx,
                        'source_document':   rid,
                        'source_page_start': pn,
                        'source_page_end':   pn,
                    })
                    global_chunk_idx += 1
            else:
                stub_text = f'[Page {pn} — no extractable text]'
                chunk_records.append({
                    'chunk_id':          f'{rid}-CHUNK-{global_chunk_idx + 1:04d}',
                    'section_id':        preamble_sid,
                    'text':              stub_text,
                    'token_count':       0,
                    'chunk_index':       global_chunk_idx,
                    'source_document':   rid,
                    'source_page_start': pn,
                    'source_page_end':   pn,
                })
                global_chunk_idx += 1
                print(f'  ⚠  Stub chunk created for preamble page {pn}')

        print(f'  Preamble chunks ({preamble_sid}): {global_chunk_idx}')

    # --- Sections: chunk from verbatim text extracted by notebook 211 ---
    # Build ordered section list for gap-filling lookup
    ordered_sections = doc_sections.sort_values('source_page_start').to_dict('records')
    sections_empty   = []
    covered_pages    = set(range(1, body_start))   # preamble pages already covered

    for row in ordered_sections:
        sid      = row['section_id']
        sec_text = str(row.get('text') or '').strip()
        p_start  = int(row['source_page_start']) if pd.notna(row.get('source_page_start')) else None
        p_end    = int(row['source_page_end'])   if pd.notna(row.get('source_page_end'))   else None

        if p_start is not None and p_end is not None:
            covered_pages.update(range(p_start, p_end + 1))

        if not sec_text:
            stub_text = f'[{sid} — no extractable text]'
            chunk_records.append({
                'chunk_id':          f'{rid}-CHUNK-{global_chunk_idx + 1:04d}',
                'section_id':        sid,
                'text':              stub_text,
                'token_count':       0,
                'chunk_index':       global_chunk_idx,
                'source_document':   rid,
                'source_page_start': p_start,
                'source_page_end':   p_end,
            })
            global_chunk_idx += 1
            sections_empty.append(sid)
            continue

        raw_chunks = chunk_text(sec_text)
        for chunk in raw_chunks:
            chunk_records.append({
                'chunk_id':          f'{rid}-CHUNK-{global_chunk_idx + 1:04d}',
                'section_id':        sid,
                'text':              chunk,
                'token_count':       len(chunk) // 4,
                'chunk_index':       global_chunk_idx,
                'source_document':   rid,
                'source_page_start': p_start,
                'source_page_end':   p_end,
            })
            global_chunk_idx += 1

    # --- Page gap check: every body page must be covered by a section's declared range ---
    # If gaps exist, sections.csv has incomplete page ranges — re-run notebook 211 to fix.
    gap_pages = sorted(set(range(body_start, total_pages + 1)) - covered_pages)
    if gap_pages:
        raise RuntimeError(
            f'{rid}: {len(gap_pages)} body page(s) not covered by any section range: {gap_pages}.\n'
            f'Re-run notebook 211 so close_page_gaps() can absorb these pages into sections.csv.'
        )

    # --- Sort by page order, preserve creation order for same source_page_start ---
    # Fixes gap-fill chunks that are created last but belong to earlier pages.
    # Tiebreaker = original creation index → co-located sections keep document order.
    indexed = sorted(
        enumerate(chunk_records),
        key=lambda x: (x[1]['source_page_start'] if x[1]['source_page_start'] is not None else 0, x[0]),
    )
    chunk_records = []
    for new_idx, (_, chunk) in enumerate(indexed):
        chunk['chunk_index'] = new_idx
        chunk['chunk_id']    = f"{rid}-CHUNK-{new_idx + 1:04d}"
        chunk_records.append(chunk)

    all_chunks.extend(chunk_records)
    print(f'  Total chunks created   : {len(chunk_records)}')
    if sections_empty:
        print(f'  Stub chunks (no text)  : {len(sections_empty)} — {sections_empty}')
    body_chunks = [c for c in chunk_records if c['section_id'] != preamble_sid]
    avg_tokens  = sum(c['token_count'] for c in body_chunks) / max(len(body_chunks), 1)
    print(f'  Avg token estimate     : {avg_tokens:.0f}')



APS-220: 30 sections
  Pages: 30 | Characters: 70,819
  Total chunks created   : 54
  Avg token estimate     : 310


## Step 4 — Write and validate

In [5]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_csv(LAYER2_DIR / 'chunks.csv', index=False)
print(f'chunks.csv : {len(chunks_df)} rows  → data/layer_2/')

# Validation — section attribution
all_section_ids = set(sections_df['section_id'].tolist())
preamble_sids   = set(chunks_df.loc[chunks_df['section_id'].str.endswith('-PREAMBLE', na=False), 'section_id'])
all_known_sids  = all_section_ids | preamble_sids

unattributed = chunks_df[chunks_df['section_id'].isna()]
if not unattributed.empty:
    print(f'\n⚠  {len(unattributed)} chunks have no section_id — THIS SHOULD NEVER HAPPEN')
else:
    print('\n✓ All chunks attributed to a section')

unknown = chunks_df[~chunks_df['section_id'].isin(all_known_sids)]
if not unknown.empty:
    print(f'⚠  {len(unknown)} chunks reference unknown section_ids')
else:
    print('✓ All section_ids are valid')

stub_chunks = chunks_df[chunks_df['token_count'] == 0]
if not stub_chunks.empty:
    print(f'⚠  {len(stub_chunks)} stub chunk(s) (no extractable text):')
    for _, r in stub_chunks.iterrows():
        print(f'    {r["chunk_id"]} | page {r["source_page_start"]} | {r["text"]}')
else:
    print('✓ No stub chunks')

# Validation — page coverage
print('\nPage coverage per document:')
for doc in docs:
    rid      = doc['regulation_id']
    pdf_path = PDF_DIR / doc['pdf_path']
    if not pdf_path.exists():
        continue
    pages       = extract_pdf_pages(pdf_path)
    total_pages = max(pn for pn, _ in pages) if pages else 0
    doc_chunks  = chunks_df[chunks_df['source_document'] == rid]

    covered = set()
    for _, r in doc_chunks.iterrows():
        if pd.notna(r['source_page_start']) and pd.notna(r['source_page_end']):
            covered.update(range(int(r['source_page_start']), int(r['source_page_end']) + 1))

    missing = set(range(1, total_pages + 1)) - covered
    if missing:
        print(f'  {rid}: ⚠  pages not covered by any chunk: {sorted(missing)}')
    else:
        print(f'  {rid}: ✓ all {total_pages} pages covered')

print('\nChunks per section:')
print(chunks_df.groupby('section_id')['chunk_id'].count().sort_values(ascending=False).to_string())

print('\nSummary per document:')
print(
    chunks_df.groupby('source_document').agg(
        chunks=('chunk_id', 'count'),
        avg_tokens=('token_count', 'mean'),
        total_tokens=('token_count', 'sum'),
    ).round(0).to_string()
)


chunks.csv : 54 rows  → data/layer_2/

✓ All chunks attributed to a section
✓ All section_ids are valid
✓ No stub chunks

Page coverage per document:
  APS-220: ✓ all 30 pages covered

Chunks per section:
section_id
APS-220-ATT-B    7
APS-220-ATT-A    5
APS-220-S17      4
APS-220-S1       3
APS-220-S6       3
APS-220-S18      3
APS-220-S23      2
APS-220-S20      2
APS-220-S21      2
APS-220-S14      2
APS-220-S15      2
APS-220-S16      1
APS-220-S27      1
APS-220-S8       1
APS-220-S7       1
APS-220-S10      1
APS-220-S5       1
APS-220-S4       1
APS-220-S3       1
APS-220-S28      1
APS-220-S25      1
APS-220-S26      1
APS-220-S24      1
APS-220-S11      1
APS-220-S22      1
APS-220-S12      1
APS-220-S13      1
APS-220-S2       1
APS-220-S19      1
APS-220-S9       1

Summary per document:
                 chunks  avg_tokens  total_tokens
source_document                                  
APS-220              54       310.0         16724
